In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import pandas as pd
import copy
import xarray as xr
import os
import re
from pathlib import Path
from torch.utils.data import Dataset, DataLoader

torch.cuda.is_available()

In [ ]:
data_dir = Path("data_fixed")
metadata_path = data_dir / "station_metadata.csv"

feature_columns = ["rain", "temp", "wetb", "dewpt", "vappr", "rhum", "msl"]
feature_units = ["mm", "°C", "°C", "°C", "hPa", "%", "hPa"]
history_hours = 24
horizon_hours = 6
block_length = pd.Timedelta(days=100)
batch_size = 64

station_metadata = pd.read_csv(metadata_path)
station_names = station_metadata["station_name"].astype(str).tolist()

# Load each station, count missing measurement values, then align stations
# on a shared hourly timeline.
station_frames = []
missing_counts = []

for station_name in station_names:
    path = data_dir / f"{station_name}.csv"
    if not path.exists():
        raise FileNotFoundError(f"Missing station file: {path}")

    df = pd.read_csv(
        path,
        usecols=["date"] + feature_columns,
        na_values=["", " ", "-", "NA", "N/A"],
    )

    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df = df.dropna(subset=["date"])

    for column in feature_columns:
        df[column] = pd.to_numeric(df[column], errors="coerce").astype("float32")

    missing_counts.append(int(df[feature_columns].isna().sum().sum()))

    df = df.set_index("date").sort_index()
    df = df[~df.index.duplicated(keep="first")]
    station_frames.append(df)

station_metadata["missing_data_points"] = missing_counts

panel = pd.concat(
    station_frames,
    axis=1,
    keys=station_names,
).sort_index()

del station_frames

In [ ]:
# Reindex to hourly timestamps so missing hours count as missing data.
hourly_index = pd.date_range(
    panel.index.min().ceil("h"),
    panel.index.max().floor("h"),
    freq="h",
)
panel = panel.reindex(hourly_index)

timestamps = panel.index
values = panel.to_numpy(dtype=np.float32, copy=False)

# A time is complete when all selected measurements at all stations exist.
complete = ~np.isnan(values).any(axis=1)

# For each start index i:
# input hours: i through i+23
# prediction target: i+23+6
target_offset = history_hours - 1 + horizon_hours
possible_starts = np.arange(max(0, len(values) - target_offset))

complete_prefix = np.concatenate(
    ([0], np.cumsum(complete, dtype=np.int64))
)
history_complete = (
    complete_prefix[possible_starts + history_hours]
    - complete_prefix[possible_starts]
) == history_hours
target_complete = complete[possible_starts + target_offset]

valid_starts = possible_starts[history_complete & target_complete]
if len(valid_starts) == 0:
    raise ValueError("No complete 24-hour input windows with a valid 6-hour target.")


In [ ]:
# Split each 100-day block by target time: 70% train, 15% validation, 15% test.
target_times = timestamps[valid_starts + target_offset]
train_parts, val_parts, test_parts = [], [], []

block_start = target_times.min().normalize()
last_target = target_times.max()

while block_start <= last_target:
    block_end = block_start + block_length
    train_cut = block_start + block_length * 0.70
    val_cut = block_start + block_length * 0.85

    in_block = (target_times >= block_start) & (target_times < block_end)

    train_parts.append(valid_starts[in_block & (target_times < train_cut)])
    val_parts.append(
        valid_starts[
            in_block & (target_times >= train_cut) & (target_times < val_cut)
        ]
    )
    test_parts.append(valid_starts[in_block & (target_times >= val_cut)])

    block_start = block_end

train_starts = np.concatenate(train_parts)
val_starts = np.concatenate(val_parts)
test_starts = np.concatenate(test_parts)


class WeatherWindows(Dataset):
    def __init__(self, values, starts):
        self.values = values
        self.starts = starts

    def __len__(self):
        return len(self.starts)

    def __getitem__(self, index):
        start = self.starts[index]
        target_index = start + target_offset

        # Copy each sample so PyTorch receives writable arrays.
        x = self.values[start : start + history_hours].copy()
        y = self.values[target_index].copy()

        return torch.from_numpy(x), torch.from_numpy(y)


train_dataset = WeatherWindows(values, train_starts)
val_dataset = WeatherWindows(values, val_starts)
test_dataset = WeatherWindows(values, test_starts)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print("Stations:", len(station_names))
print("Missing measurement values by station:")
display(station_metadata[["station_name", "missing_data_points"]])

print("Train windows:", len(train_dataset))
print("Validation windows:", len(val_dataset))
print("Test windows:", len(test_dataset))
print(
    "Input shape per sample:",
    (history_hours, len(station_names), len(feature_columns)),
)
print("Target shape per sample:", (len(station_names), len(feature_columns)))

# Example batch shapes:
X_batch, Y_batch = next(iter(train_loader))
print("Batch input shape:", X_batch.shape)
print("Batch target shape:", Y_batch.shape)

# Persistence

In [ ]:
def make_full_tensors(dataset):
    X = np.stack([
        dataset[i][0].numpy()
        for i in range(len(dataset))
    ])
    Y = np.stack([
        dataset[i][1].numpy()
        for i in range(len(dataset))
    ])
    return torch.from_numpy(X), torch.from_numpy(Y)

X_train, Y_train = make_full_tensors(train_dataset)
X_val, Y_val = make_full_tensors(val_dataset)
X_test, Y_test = make_full_tensors(test_dataset)

In [ ]:
# Persistence baseline: use the last observed hour to predict the value 6 hours later.
# Fit one mean/std per feature using training data only, shared across stations.
# X_train and X_test are currently flattened as [samples, history, stations * features].

num_stations = len(station_names)
num_features = len(feature_columns)

X_train_4d = X_train.reshape(
    -1, history_hours, num_stations, num_features
)
X_test_4d = X_test.reshape(
    -1, history_hours, num_stations, num_features
)
Y_test_3d = Y_test.reshape(
    -1, num_stations, num_features
)

feature_mean = X_train_4d.mean(dim=(0, 1, 2), keepdim=True)
feature_std = X_train_4d.std(dim=(0, 1, 2), keepdim=True).clamp_min(1e-6)

# Scale prediction and target with the same training-set statistics.
X_test_scaled = (X_test_4d - feature_mean) / feature_std
Y_test_scaled = (
    Y_test_3d - feature_mean.reshape(1, 1, num_features)
) / feature_std.reshape(1, 1, num_features)

# Persistence forecast: value at the final input hour, predicting 6 hours ahead.
Y_pred_persistence_scaled = X_test_scaled[:, -1, :, :]

# Calculate one RMSE per feature, aggregating across samples and stations.
persistence_mse_scaled_per_feature = (
    (Y_test_scaled - Y_pred_persistence_scaled) ** 2
).mean(dim=(0, 1))

persistence_rmse_per_feature = (
    torch.sqrt(persistence_mse_scaled_per_feature) * feature_std.reshape(-1)
)

for feature, unit, rmse_value in zip(
    feature_columns, feature_units, persistence_rmse_per_feature.tolist()
):
    print(f"{feature}: persistence RMSE = {rmse_value:.3f} {unit}")

# GNN

In [ ]:
from torch import nn
from sklearn.preprocessing import StandardScaler

try:
    from torch_geometric.data import Data
    from torch_geometric.nn import GCNConv, GATConv
    PYG_OK = True
except Exception as e:
    PYG_OK = False
    print("PyTorch Geometric not found. GCN/GAT sections will error if you run them. See install instructions above.")


In [ ]:
# Build directed edges between every pair of stations.
# Edge direction is source -> destination:
#   distance_m       = distance from source to destination
#   angle_deg        = clockwise bearing from north at source to destination
#   height_change_m  = destination height - source height
#
# Rows in station_metadata must be in the same order as the stations in your
# node data. This creates both directions for every station pair, but no
# self-edges.

coords = station_metadata[
    ["latitude", "longitude", "station_height_m"]
].to_numpy(dtype=np.float64)

if not np.isfinite(coords).all():
    raise ValueError("Station coordinates or heights contain missing values.")

num_stations = len(coords)
edge_pairs = []
edge_values = []

earth_radius_m = 6_371_000.0

for source in range(num_stations):
    lat1, lon1, height1 = np.radians(coords[source, :2]).tolist() + [coords[source, 2]]

    for destination in range(num_stations):
        if source == destination:
            continue

        lat2, lon2, height2 = (
            np.radians(coords[destination, :2]).tolist()
            + [coords[destination, 2]]
        )

        delta_lat = lat2 - lat1
        delta_lon = lon2 - lon1

        # Haversine distance, in metres.
        a = (
            np.sin(delta_lat / 2) ** 2
            + np.cos(lat1) * np.cos(lat2) * np.sin(delta_lon / 2) ** 2
        )
        distance_m = 2 * earth_radius_m * np.arcsin(np.sqrt(a))

        # Initial bearing clockwise from north, in degrees [0, 360).
        y = np.sin(delta_lon) * np.cos(lat2)
        x = (
            np.cos(lat1) * np.sin(lat2)
            - np.sin(lat1) * np.cos(lat2) * np.cos(delta_lon)
        )
        angle_deg = (np.degrees(np.arctan2(y, x)) + 360) % 360

        height_change_m = height2 - height1

        edge_pairs.append([source, destination])
        edge_values.append([distance_m, angle_deg, height_change_m])

# Shapes:
# edge_index: [2, number_of_edges]
# edge_attr:  [number_of_edges, 3], ordered as distance, angle, height change
edge_index = torch.tensor(edge_pairs, dtype=torch.long).T.contiguous()
edge_attr = torch.tensor(edge_values, dtype=torch.float32)

print("edge_index shape:", edge_index.shape)
print("edge_attr shape:", edge_attr.shape)

In [ ]:
# This cell uses the variables and data already created above:
# edge_index, edge_attr, X_train, train_loader, val_loader, test_loader,
# station_names, feature_columns, feature_units, history_hours.
#
# The data loaders return flattened station-feature dimensions:
# X_batch: [batch, time, stations * features]
# Y_batch: [batch, stations * features]
# The model reshapes these internally to:
# X: [batch, time, stations, features]
# Y: [batch, stations, features]

from torch_geometric.nn import GATv2Conv

num_stations = len(station_names)
num_features = len(feature_columns)


def make_knn_graph(edge_index, edge_attr, num_stations, k):
    """Create directed edges from each station's k nearest neighbours."""
    if not 1 <= k < num_stations:
        raise ValueError(f"k must be between 1 and {num_stations - 1}")

    source, destination = edge_index
    knn_edges = []
    knn_attributes = []

    for dest in range(num_stations):
        candidate_edges = torch.where(destination == dest)[0]
        closest_edges = candidate_edges[
            torch.topk(
                edge_attr[candidate_edges, 0],
                k=k,
                largest=False,
            ).indices
        ]

        source_nodes = source[closest_edges]
        selected_attr = edge_attr[closest_edges]

        # Scale distance and height; encode bearing as sin/cos to handle
        # the circular boundary between 359° and 0°.
        bearing = torch.deg2rad(selected_attr[:, 1])
        selected_attr_scaled = torch.stack(
            [
                selected_attr[:, 0] / 10_000.0,  # distance, in 10 km units
                torch.sin(bearing),
                torch.cos(bearing),
                selected_attr[:, 2] / 1_000.0,   # height change, in km
            ],
            dim=1,
        )

        destination_nodes = torch.full_like(source_nodes, dest)
        knn_edges.append(torch.stack([source_nodes, destination_nodes]))
        knn_attributes.append(selected_attr_scaled)

    return torch.cat(knn_edges, dim=1), torch.cat(knn_attributes, dim=0)


class TemporalSpatialGNN(nn.Module):
    def __init__(
        self,
        num_stations,
        num_features,
        hidden_dim=64,
        num_heads=4,
        num_gat_layers=2,
        dropout=0.1,
    ):
        super().__init__()

        self.num_stations = num_stations
        self.num_features = num_features

        # Encode the 24-hour history separately for each station.
        self.temporal_encoder = nn.GRU(
            input_size=num_features,
            hidden_size=hidden_dim,
            batch_first=True,
        )

        # Each GATv2 layer attends over the station's graph neighbours,
        # using distance, bearing, and height change as edge attributes.
        self.gat_layers = nn.ModuleList([
            GATv2Conv(
                in_channels=hidden_dim,
                out_channels=hidden_dim,
                heads=num_heads,
                concat=False,
                edge_dim=4,
                dropout=dropout,
                add_self_loops=False,
            )
            for _ in range(num_gat_layers)
        ])

        self.norms = nn.ModuleList([
            nn.LayerNorm(hidden_dim) for _ in range(num_gat_layers)
        ])
        self.dropout = nn.Dropout(dropout)
        self.output_layer = nn.Linear(hidden_dim, num_features)

    def forward(self, x, edge_index, edge_attr):
        # Accept the flattened batches produced by WeatherWindows:
        # [B, T, N*F]. Also accept already reshaped [B, T, N, F] input.
        if x.ndim == 3:
            batch_size, history, flattened_features = x.shape
            expected = self.num_stations * self.num_features
            if flattened_features != expected:
                raise ValueError(
                    f"Expected {expected} flattened station features, "
                    f"got {flattened_features} in shape {tuple(x.shape)}"
                )
            x = x.reshape(
                batch_size, history, self.num_stations, self.num_features
            )
        elif x.ndim == 4:
            batch_size, history, stations, features = x.shape
            if (stations, features) != (self.num_stations, self.num_features):
                raise ValueError(f"Unexpected input shape: {tuple(x.shape)}")
        else:
            raise ValueError(
                "Expected input shaped [B,T,N*F] or [B,T,N,F], "
                f"got {tuple(x.shape)}"
            )

        # GRU input: [batch * stations, time, features].
        x = x.permute(0, 2, 1, 3).reshape(
            batch_size * self.num_stations,
            history,
            self.num_features,
        )
        _, hidden = self.temporal_encoder(x)
        x = hidden[-1]

        # Replicate the station graph for each example in the batch.
        edges_per_graph = edge_index.shape[1]
        offsets = (
            torch.arange(batch_size, device=x.device)
            .repeat_interleave(edges_per_graph)
            * self.num_stations
        )
        batched_edge_index = (
            edge_index.repeat(1, batch_size) + offsets.unsqueeze(0)
        )
        batched_edge_attr = edge_attr.repeat(batch_size, 1)

        # Spatial message passing with residual connections.
        for gat, norm in zip(self.gat_layers, self.norms):
            updated = gat(x, batched_edge_index, batched_edge_attr)
            x = norm(x + self.dropout(updated))

        return self.output_layer(x).reshape(
            batch_size, self.num_stations, self.num_features
        )




In [ ]:
# Set k to control how many nearest neighbours each station connects to.
k = 4
edge_index_knn, edge_attr_knn = make_knn_graph(
    edge_index=edge_index,
    edge_attr=edge_attr,
    num_stations=num_stations,
    k=k,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
edge_index_knn = edge_index_knn.to(device)
edge_attr_knn = edge_attr_knn.to(device)

model = TemporalSpatialGNN(
    num_stations=num_stations,
    num_features=num_features,
    hidden_dim=64,
    num_heads=4,
    num_gat_layers=2,
    dropout=0.1,
).to(device)

# Calculate one training-set mean and standard deviation per feature,
# shared across stations. reshape() is a view of the existing tensor.
X_train_4d = X_train.reshape(
    -1, history_hours, num_stations, num_features
)
feature_mean = X_train_4d.mean(
    dim=(0, 1, 2), keepdim=True
).to(device)
feature_std = X_train_4d.std(
    dim=(0, 1, 2), keepdim=True
).clamp_min(1e-6).to(device)
del X_train_4d

target_mean = feature_mean.reshape(1, 1, num_features)
target_std = feature_std.reshape(1, 1, num_features)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-5,
)
loss_function = nn.MSELoss()
num_epochs = 20

# Set patience to the number of consecutive epochs without sufficient
# validation-loss improvement before stopping.
patience = 5
min_delta = 1e-4

train_losses = []
val_losses = []
best_val_loss = float("inf")
best_epoch = 0
epochs_without_improvement = 0
best_model_state = None

for epoch in range(num_epochs):
    model.train()
    train_loss_sum = 0.0
    train_count = 0

    for X_batch, Y_batch in train_loader:
        batch_size_now = X_batch.shape[0]

        X_batch = X_batch.reshape(
            batch_size_now, history_hours, num_stations, num_features
        ).to(device)
        Y_batch = Y_batch.reshape(
            batch_size_now, num_stations, num_features
        ).to(device)

        X_batch = (X_batch - feature_mean) / feature_std
        Y_batch = (Y_batch - target_mean) / target_std

        optimizer.zero_grad()
        predictions = model(X_batch, edge_index_knn, edge_attr_knn)
        loss = loss_function(predictions, Y_batch)
        loss.backward()
        optimizer.step()

        train_loss_sum += loss.item() * batch_size_now
        train_count += batch_size_now

    train_epoch_loss = train_loss_sum / train_count
    train_losses.append(train_epoch_loss)

    model.eval()
    val_loss_sum = 0.0
    val_count = 0

    with torch.no_grad():
        for X_batch, Y_batch in val_loader:
            batch_size_now = X_batch.shape[0]

            X_batch = X_batch.reshape(
                batch_size_now, history_hours, num_stations, num_features
            ).to(device)
            Y_batch = Y_batch.reshape(
                batch_size_now, num_stations, num_features
            ).to(device)

            X_batch = (X_batch - feature_mean) / feature_std
            Y_batch = (Y_batch - target_mean) / target_std

            predictions = model(X_batch, edge_index_knn, edge_attr_knn)
            loss = loss_function(predictions, Y_batch)

            val_loss_sum += loss.item() * batch_size_now
            val_count += batch_size_now

    val_epoch_loss = val_loss_sum / val_count
    val_losses.append(val_epoch_loss)

    if val_epoch_loss < best_val_loss - min_delta:
        best_val_loss = val_epoch_loss
        best_epoch = epoch + 1
        epochs_without_improvement = 0
        best_model_state = copy.deepcopy(model.state_dict())
    else:
        epochs_without_improvement += 1

    print(
        f"Epoch {epoch + 1}/{num_epochs} | "
        f"train MSE: {train_epoch_loss:.4f} | "
        f"validation MSE: {val_epoch_loss:.4f} | "
        f"patience: {epochs_without_improvement}/{patience}"
    )

    if epochs_without_improvement >= patience:
        print(f"Early stopping. Best epoch: {best_epoch}")
        break

# Restore the model weights from the epoch with the best validation loss.
if best_model_state is not None:
    model.load_state_dict(best_model_state)

print(f"Best validation MSE: {best_val_loss:.4f} at epoch {best_epoch}")

# Plot training and validation losses.
plt.figure(figsize=(8, 5))
epochs = range(1, len(train_losses) + 1)
plt.plot(epochs, train_losses, label="Training loss")
plt.plot(epochs, val_losses, label="Validation loss")
plt.axvline(best_epoch, color="gray", linestyle="--", label="Best epoch")
plt.xlabel("Epoch")
plt.ylabel("Normalized MSE loss")
plt.title("Training and validation loss")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Replot the graph with a dedicated colorbar area and labels kept inside
# the map axes, away from the colorbar.

import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable

positions = station_metadata[["longitude", "latitude"]].to_numpy()
names = station_metadata["station_name"].astype(str).tolist()

edges = edge_index_knn.detach().cpu().numpy()
attrs = edge_attr_knn.detach().cpu().numpy()
distances_km = attrs[:, 0] * 10

norm = Normalize(vmin=distances_km.min(), vmax=distances_km.max())
cmap = plt.colormaps["viridis"]
edge_set = set(zip(edges[0], edges[1]))

fig = plt.figure(figsize=(14, 9))
ax = fig.add_axes([0.05, 0.08, 0.76, 0.84])
cax = fig.add_axes([0.87, 0.20, 0.025, 0.60])

for i, (src, dst) in enumerate(zip(edges[0], edges[1])):
    x1, y1 = positions[src]
    x2, y2 = positions[dst]

    reciprocal = (dst, src) in edge_set
    curve = 0.12 if reciprocal and src < dst else (
        -0.12 if reciprocal else 0.0
    )

    ax.add_patch(FancyArrowPatch(
        (x1, y1),
        (x2, y2),
        arrowstyle="-|>",
        mutation_scale=10,
        linewidth=1.2,
        color=cmap(norm(distances_km[i])),
        alpha=0.7,
        connectionstyle=f"arc3,rad={curve}",
        shrinkA=8,
        shrinkB=8,
        zorder=1,
    ))

ax.scatter(
    positions[:, 0],
    positions[:, 1],
    s=75,
    color="darkorange",
    edgecolor="black",
    zorder=2,
)

# Put labels on the side of each point that faces away from the colorbar.
mid_longitude = positions[:, 0].mean()
for i, name in enumerate(names):
    on_eastern_side = positions[i, 0] > mid_longitude
    ax.annotate(
        name.replace("_", " "),
        positions[i],
        xytext=(-7, 5) if on_eastern_side else (7, 5),
        textcoords="offset points",
        ha="right" if on_eastern_side else "left",
        fontsize=8,
        clip_on=True,
        zorder=3,
    )

fig.colorbar(
    ScalarMappable(norm=norm, cmap=cmap),
    cax=cax,
    label="Edge distance (km)",
)

ax.set_title(f"Directed {k}-nearest-neighbour graph")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.grid(True, alpha=0.25)
ax.set_aspect("equal", adjustable="datalim")
plt.show()

In [ ]:
print(feature_std)

In [ ]:
for feature, unit, rmse_value in zip(
    feature_columns, feature_units, persistence_rmse_per_feature.tolist()
):
    print(f"{feature}: persistence RMSE = {rmse_value:.3f} {unit}")

In [ ]:
# Evaluate the trained model on the test set and report RMSE in original units.
model.eval()
test_squared_error = torch.zeros(num_features, device=device)
test_count = 0

with torch.no_grad():
    for X_batch, Y_batch in test_loader:
        batch_size_now = X_batch.shape[0]

        X_batch = X_batch.reshape(
            batch_size_now, history_hours, num_stations, num_features
        ).to(device)
        Y_batch = Y_batch.reshape(
            batch_size_now, num_stations, num_features
        ).to(device)

        X_batch = (X_batch - feature_mean) / feature_std
        Y_batch = (Y_batch - target_mean) / target_std

        predictions = model(X_batch, edge_index_knn, edge_attr_knn)
        test_squared_error += ((predictions - Y_batch) ** 2).sum(dim=(0, 1))
        test_count += batch_size_now * num_stations

test_rmse = (
    (test_squared_error / test_count).sqrt()
    * feature_std.reshape(-1)
)

for feature, unit, rmse_value in zip(
    feature_columns, feature_units, test_rmse.cpu().tolist()
):
    print(f"{feature}: test RMSE = {rmse_value:.3f} {unit}")

In [ ]:
fig, axs = plt.subplots(1,1, figsize = (10,10))
w, x = 0.4, np.arange(len(feature_columns))
axs.bar(x + w/2, test_rmse.cpu().tolist(), width = w, color="steelblue", edgecolor="black")
axs.bar(x - w/2, persistence_rmse_per_feature.cpu().tolist(), width = w, color="orange", edgecolor="black")

axs_labels = [f"{f}\n({u})" for f, u in zip(feature_columns, feature_units)]

axs.set_title("Test RMSE by Feature")
axs.set_xticks(x)
axs.set_xticklabels(axs_labels)
axs.set_xlabel("Feature")
axs.set_ylabel("RMSE (original units)")
axs.legend(["GNN Model", "Persistence Baseline"])
plt.show()

In [ ]:
print(predictions.shape)
print(X_batch.shape)